In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

In [17]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
'''Your task is to translate a textual rule into an equivalent SPARQL query so that whenever the rule is violated, the result set of the query is non-empty and vice versa. 
Please formulate a SELECT-WHERE query. 
Create a query that includes the variable "?case", which relates to one specific process instance.
Please only output the textual rule under the header "rule" and the SPARQL query under the header "query". Don't output any notes or justifications.

For generating the rules, please consider the following contextual schema information in OWL-RDF format and key entities in RDF format.  

{context}''',
        ),
        ("human", "Please translate the following rule {rule}"),
    ]
)

In [18]:
context = '''
@prefix relation: <http://example.org/relations/> .
@prefix type: <http://example.org/types/> .
@prefix sepsis: <http://example.org/sepsis/> .

type:Task rdf:type owl:Class ;
    rdfs:label "Task" ;
    rdfs:isDefinedBy "One specific task to be performed".

type:Case rdf:type owl:Class ;
    rdfs:label "Case" ;
    rdfs:isDefinedBy "One instance of a process such as one shopping order".

type:Activity rdf:type owl:Class ;
    rdfs:label "Activity" ;    
    rdfs:isDefinedBy "A recurring activity of a process".

relation:instanceOf rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain type:Task ;
    rdfs:range type:Activity .

relation:partOf rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain type:Task ;
    rdfs:range type:Case .


sepsis:Measurement rdf:type owl:Class ;
    rdfs:label "Measurement" ;    
    rdfs:isDefinedBy "A measurement of a clinical parameter".

sepsis:Clinical_Parameter rdf:type owl:Class ;
    rdfs:label "Clinical Parameter" ;    
    rdfs:isDefinedBy "A clinical parameter".

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:mes_value rdf:type owl:DatatypeProperty, owl:FunctionalProperty ;    
    rdfs:isDefinedBy "The value of a measurement" ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range xsd:decimal .

sepsis:produced rdf:type owl:ObjectProperty, owl:FunctionalProperty ;    
    rdfs:isDefinedBy "A measurement has been produced" ;
    rdfs:domain type:Task ;
    rdfs:range sepsis:Measurement .

sepsis:leucocyte_count a sepsis:Clinical_Parameter ;
    rdfs:label "Leucocyte Count" ;    
    rdfs:isDefinedBy "The number of Leucocytes (white blood cells) in 10^9 per liter of blood.".
    
'''

In [19]:
chain = prompt | llm
response = chain.invoke(
    {
        "context": context,
        "rule": "Leucocyte measurements must be between 4.0 and 12.0 million.",
    }
)

print(response.content)

In [21]:
print(response.content)

rule  
Leucocyte measurements must be between 4.0 and 12.0 million.

query  
SELECT ?case  
WHERE {  
  ?task relation:partOf ?case .  
  ?task sepsis:produced ?measurement .  
  ?measurement sepsis:measurand sepsis:leucocyte_count .  
  ?measurement sepsis:mes_value ?value .  
  FILTER(?value < 4.0 || ?value > 12.0)  
}  
